# Can we predict late deliveries?
Target: `late_delivery` (1 = delivered after promised date). Unit: one consignment.

**Key decision – `claim_paid_eur` is excluded.** It is compensation paid on a customer claim *settled for the consignment*, i.e. it is only known after delivery and is almost a restatement of the target (checked below). A model using it would look great but couldn't be used at dispatch time, which is when a prediction is useful. Features used: only what is known at dispatch (`dispatched_on`, `carrier`, `distance_km`, `weight_kg`, `parcels`).

Validation: chronological split (train on earlier dispatches, test on the latest ones), since the model would be deployed on future consignments.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, RocCurveDisplay
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
df = pd.read_csv('data.csv', parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
print(df.shape, df.consignment.is_unique, df.isna().sum().sum(), 'missing')
print(df.dispatched_on.min(), '->', df.dispatched_on.max())
df.describe().T

## 1. Data checks and leakage

In [ ]:
print('Late rate:', df.late_delivery.mean().round(3))
print(df.groupby(df.claim_paid_eur > 0).late_delivery.agg(['mean','size']).rename(index={False:'no claim paid',True:'claim paid'}))
print(df.groupby('carrier').late_delivery.agg(['mean','size']))
print('Late rate by month:'); print(df.groupby(df.dispatched_on.dt.to_period('M')).late_delivery.agg(['mean','size']).T.round(3).to_string())

Claims are paid on ~90% of late consignments and on **none** of the on-time ones, so `claim_paid_eur` is a post-outcome leak and is dropped.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
for a, c in zip(ax, ['distance_km','weight_kg','parcels']):
    q = pd.qcut(df[c], 10, duplicates='drop')
    df.groupby(q, observed=True).late_delivery.mean().plot(marker='o', ax=a)
    a.set_title(f'Late rate by {c} decile'); a.set_xlabel(''); a.tick_params(axis='x', rotation=60)
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
df.groupby(df.dispatched_on.dt.hour).late_delivery.mean().plot(marker='o', ax=ax[0], title='Late rate by dispatch hour')
df.groupby(df.dispatched_on.dt.dayofweek).late_delivery.mean().plot(marker='o', ax=ax[1], title='Late rate by weekday (0=Mon)')
plt.tight_layout(); plt.show()

## 2. Features and chronological split

In [ ]:
X = df[['carrier','distance_km','weight_kg','parcels']].copy()
X['hour'] = df.dispatched_on.dt.hour
X['dow'] = df.dispatched_on.dt.dayofweek
X['month'] = df.dispatched_on.dt.month
y = df.late_delivery
cut = int(len(df) * 0.8)
Xtr, Xte, ytr, yte = X[:cut], X[cut:], y[:cut], y[cut:]
print('train', Xtr.shape, ytr.mean().round(3), '| test', Xte.shape, yte.mean().round(3))
print('test period starts', df.dispatched_on[cut])

In [ ]:
num = ['distance_km','weight_kg','parcels','hour','dow','month']
pre = ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), ['carrier']), ('n', StandardScaler(), num)])
models = {
 'baseline (prior)': DummyClassifier(strategy='prior'),
 'logistic regression': Pipeline([('p', pre), ('m', LogisticRegression(max_iter=1000))]),
 'gradient boosting': Pipeline([('p', ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), ['carrier']), ('n','passthrough',num)])),
                                ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))]),
}
rows, probs = [], {}
for n, m in models.items():
    cv = cross_val_score(m, Xtr, ytr, cv=TimeSeriesSplit(5), scoring='roc_auc')
    m.fit(Xtr, ytr); p = m.predict_proba(Xte)[:,1]; probs[n] = p
    rows.append(dict(model=n, cv_auc_train=cv.mean(), test_auc=roc_auc_score(yte, p),
                     test_pr_auc=average_precision_score(yte, p), test_brier=brier_score_loss(yte, p)))
res = pd.DataFrame(rows).set_index('model').round(3); print('Test late rate (PR-AUC baseline):', yte.mean().round(3)); res

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
for n in list(models)[1:]: RocCurveDisplay.from_predictions(yte, probs[n], name=n, ax=ax)
ax.plot([0,1],[0,1],'k--'); plt.show()
# Lift: late rate by predicted-risk quintile of the best model (chosen by CV, not test)
best = res.cv_auc_train.idxmax(); q = pd.qcut(probs[best], 5, labels=False, duplicates='drop')
print(best); print(pd.DataFrame({'q':q,'y':yte.values}).groupby('q').y.agg(['mean','size']).round(3))

## 3. What drives lateness?

In [ ]:
from sklearn.inspection import permutation_importance
pi = permutation_importance(models[best], Xte, yte, scoring='roc_auc', n_repeats=10, random_state=0)
pd.Series(pi.importances_mean, index=X.columns).sort_values().plot.barh(title='Permutation importance (drop in test AUC)'); plt.show()

## 4. Findings
Results from the last run (rows of the table above): logistic regression CV AUC 0.72 / test AUC 0.59 / PR-AUC 0.052 vs. 0.022 base rate; gradient boosting CV 0.71 / test 0.59.

- **The late rate is not stationary.** It hovers ~10-45% until Sep 2025, then collapses to 1-5% from Oct 2025 (the whole test window: 2.2% late, ~35 positives). Either operations changed, or recent consignments' outcomes are not fully recorded yet (right-censoring). This must be confirmed with the data owner; it dominates every result.
- Within the training period dispatch-time features carry modest signal (CV AUC ~0.72), but on the recent window it falls to ~0.59, so the test estimate is both weak and very noisy.
- Carrier differs (coastal ~27% late vs express ~18%, north ~19%).

## 4. Caveats
- Only dispatch-time features exist; no weather, route, depot load or promised-date information, which likely limits achievable accuracy.
- `claim_paid_eur` was deliberately excluded (leakage). With it the problem is trivial but useless at dispatch time.
- Single chronological hold-out (last 20% of dispatches); performance on later periods may drift.